# Dental Panoramic AI — IMPROVEMENT RUN v2 (Colab)

**Project:** Tufts + DENTEX cross-dataset panoramic X-ray AI
([boakyejeff/dental-panoramic-ai](https://github.com/boakyejeff/dental-panoramic-ai)).
**What this notebook does:** tries to beat run 1 (2026-11-07, free Tesla T4)
with three targeted changes — each one aims at a specific run-1 failure —
and the evaluation is IDENTICAL to run 1 (DENTEX val 50 images / 182 boxes;
Tufts test 151), so the numbers compare cleanly.

**Run-1 baseline (from `results/detector_metrics_gpu.json` etc.):**
- Detector (ResNet50-FPN unfrozen, 1024×512, 5 epochs): mAP@0.5 **0.355**
  (0.377 @0.3); Caries 0.28, Deep Caries 0.34, Impacted 0.69,
  Periapical Lesion 0.11 (regressed from CPU 0.29); Tufts cross-site
  sensitivity 0.90 / specificity **0.09**.
- Classifier (mixed-site ResNet18, seed 42): DENTEX val AUC **0.554**
  (did NOT reproduce the CPU v2 run's 0.84), Tufts test AUC 0.587.

**V2 changes:**
1. **Run A — detector + 200 Tufts normal images as box-free negatives, 10 epochs.**
   Run 1's specificity collapse (0.09) is structural: DENTEX training has
   only 27 normal images in 705. Negatives come only from the Tufts TRAIN
   split — val/test images are excluded, so no eval image is trained on.
   Loss was still falling at epoch 5 (0.599 → 0.307), so 10 epochs.
2. **Run B — same as A + focal loss on the ROI classification head**
   (gamma 2.0, alpha 0.25). Targets the periapical regression.
   Only if time permits in this session; same evaluation.
3. **Classifier replication — the same mixed-site recipe with 3 seeds
   (42, 0, 1)**, plus per-site (oracle) thresholds and a cross-site
   threshold table. Settles whether the CPU 0.84 or the GPU 0.554 is
   the fluke.

**Before you run anything:**
1. **Runtime > Change runtime type > T4 GPU.** Free tier, no Pro purchase.
2. The Tufts zips are ALREADY in Drive (`MyDrive/dental-panoramic-ai/`)
   from run 1 — this notebook checks and reuses them. DENTEX downloads
   itself from Zenodo.
3. This notebook clones the repo from GitHub — the v2 scripts must be
   pushed first (they are, if you are reading this from the GitHub copy).

In [ ]:
# 1 — Confirm the GPU. If this says "No GPU", stop:
# Runtime > Change runtime type > T4 GPU, then run this cell again.
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 2 — Mount Google Drive

This is where the Tufts zips come from and where all results go, so a
Colab disconnect does not lose the run.


In [ ]:
# 2 — Mount Drive (sign in with the Google account that has the Tufts zips).
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")


## 3 — Get the project code

Clones the GitHub repo (public, no login needed). If you edited the
scripts locally since the last push, push first — Colab always uses
what is on GitHub, not your local disk.


In [ ]:
# 3 — Clone the repo (or pull the latest if it is already here).
import os
PROJECT = "/content/dental-panoramic-ai"
if os.path.isdir(PROJECT):
    !cd {PROJECT} && git pull --ff-only
else:
    !git clone https://github.com/boakyejeff/dental-panoramic-ai.git {PROJECT}
%cd {PROJECT}
!ls
assert os.path.isfile("colab/train_detector_v2.py"), "v2 detector script missing — check the clone"
assert os.path.isfile("colab/train_classifier_v2.py"), "v2 classifier script missing — check the clone"
print("Project ready at", PROJECT)


## 4 — Install dependencies

**Do NOT `pip install -r requirements.txt` here.** That file pins
`torch==2.14.0+cpu`, which would replace Colab's CUDA PyTorch with the
CPU build and silently put training back on the CPU. Colab already
ships CUDA PyTorch — install only the small packages.


In [ ]:
# 4 — Small packages only. Colab's CUDA torch is kept as-is.
!pip install -q numpy pandas scikit-learn matplotlib Pillow
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Still no GPU — set Runtime > Change runtime type > T4 GPU"


## 5 — Download DENTEX (Zenodo, public, no login)

Training archive is 10.9 GB — the download takes ~5–15 min in Colab.
MD5s are checked against the Zenodo record (7812323); a mismatch stops
the notebook, because a corrupt 10.9 GB download must not train silently.


In [ ]:
# 5 — DENTEX from Zenodo (record 7812323), with MD5 verification.
import hashlib, os, subprocess
DATA = "data"
os.makedirs(DATA, exist_ok=True)

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

FILES = [
    ("training_data.zip", "https://zenodo.org/api/records/7812323/files/training_data.zip/content",
     "eba0c44fa4c6e8ec221db3009eccc1f9", 10927391529),
    ("validation_data.zip", "https://zenodo.org/api/records/7812323/files/validation_data.zip/content",
     "b2a13267740185e42cb59e968bc3ce83", 149512256),
]
for name, url, want_md5, want_size in FILES:
    p = os.path.join(DATA, name)
    if os.path.exists(p) and os.path.getsize(p) == want_size and md5(p) == want_md5:
        print(f"{name}: already present and MD5-verified, skipping download")
        continue
    print(f"downloading {name} ...")
    subprocess.run(["curl", "-L", "-o", p, url], check=True)
    assert os.path.getsize(p) == want_size, f"{name}: size mismatch"
    assert md5(p) == want_md5, f"{name}: MD5 mismatch — re-download"
    print(f"{name}: OK ({want_size} bytes, MD5 verified)")

# Unzip (training zip contains training_data/ + the triple JSONs).
subprocess.run(["unzip", "-q", "-o", f"{DATA}/training_data.zip", "-d", f"{DATA}/training"], check=True)
subprocess.run(["unzip", "-q", "-o", f"{DATA}/validation_data.zip", "-d", f"{DATA}/validation_data"], check=True)
# Validation triple JSON (COCO-like; the local scripts read it as validation_triple2.json).
subprocess.run(["curl", "-sL", "-o", f"{DATA}/validation_triple2.json",
                "https://raw.githubusercontent.com/ibrahimethemhamamci/DENTEX/main/DENTEX/validation_triple.json"],
               check=True)
!ls data/training/training_data/ && ls data/validation_data/ | head
print("DENTEX ready.")


## 6 — Tufts data

**There is no scriptable direct download for Tufts.** The Tufts site
(http://tdd.ece.tufts.edu/) is a JavaScript page and its zip links
return 403 to scripts, so this cell does not pretend to download it.
Use one of these instead:

- **Recommended:** upload `Radiographs.zip` (119 MB) and `Expert.zip`
  (49 MB) — the files already on the local machine — to
  `MyDrive/dental-panoramic-ai/` in Google Drive, then run this cell.
  Run 1 already uploaded them — if they are there, this cell just
  reuses them; do not re-upload.
- **Alternative:** the Kaggle mirror
  (`deepologylab/tufts-dental-database`) with a Kaggle API token
  (`kaggle.json` in Drive or `~/.kaggle/`). Needs a Kaggle account.


In [ ]:
# 6 — Tufts from Drive (recommended), Kaggle mirror as fallback.
import os, shutil, subprocess, glob

CANDIDATES = [
    "/content/drive/MyDrive/dental-panoramic-ai",
    "/content/drive/MyDrive/dental-panoramic-ai/data",
    "/content/drive/MyDrive",
]
found = {}
for d in CANDIDATES:
    for name in ("Radiographs.zip", "Expert.zip"):
        p = os.path.join(d, name)
        if os.path.isfile(p):
            found.setdefault(name, p)
missing = [n for n in ("Radiographs.zip", "Expert.zip") if n not in found]
if missing:
    # Kaggle fallback (needs ~/.kaggle/kaggle.json or KAGGLE_USERNAME/KEY).
    print(f"Not in Drive: {missing}. Trying Kaggle mirror ...")
    r = subprocess.run(["kaggle", "datasets", "download", "-d",
                        "deepologylab/tufts-dental-database", "-p", "data", "--unzip"],
                       capture_output=True, text=True)
    print(r.stdout[-500:], r.stderr[-500:])
    if r.returncode != 0:
        raise SystemExit(
            "Tufts zips not found. Upload Radiographs.zip + Expert.zip to "
            "MyDrive/dental-panoramic-ai/ in Google Drive and re-run this cell, "
            "or set up a Kaggle API token for the deepologylab mirror.")
for name, src in found.items():
    dst = os.path.join("data", name)
    if not os.path.exists(dst):
        print(f"copying {src} -> {dst}")
        shutil.copy(src, dst)
    print(name, os.path.getsize(dst), "bytes")
subprocess.run(["unzip", "-q", "-o", "data/Radiographs.zip", "-d", "data/Radiographs"], check=True)
subprocess.run(["unzip", "-q", "-o", "data/Expert.zip", "-d", "data/Expert"], check=True)
print("Tufts ready:", len(glob.glob("data/Radiographs/*.JPG")), "radiographs")


## 7 — Build resized image caches

The training scripts read resized JPEG caches, not the raw PNGs/JPGs.
1024×512 is what the GPU detector trains on; 512×256 is what the
classifier uses. ~5 min.


In [ ]:
# 7 — Write the cache builder, then run it.
%%writefile colab/build_caches.py
"""Build 1024x512 + 512x256 JPEG caches (Colab). Skips files already built."""
import json
from pathlib import Path
from PIL import Image

BASE = Path(__file__).resolve().parent.parent
DATA = BASE / "data"

def cache_dir(src_dir, pattern, outdir, size):
    outdir.mkdir(parents=True, exist_ok=True)
    files = sorted(Path(src_dir).glob(pattern))
    done = 0
    for p in files:
        dst = outdir / (p.stem + ".jpg")
        if dst.exists():
            done += 1
            continue
        Image.open(p).convert("RGB").resize(size, Image.LANCZOS).save(dst, quality=90)
        done += 1
    print(f"{size}: {done} cached in {outdir}")

# DENTEX train (705 PNGs, from the training triple JSON)
d = json.load(open(DATA / "training" / "training_data" /
                   "quadrant-enumeration-disease" / "train_quadrant_enumeration_disease.json"))
names = sorted({i["file_name"] for i in d["images"]})
src = DATA / "training" / "training_data" / "quadrant-enumeration-disease" / "xrays"
for size, out in [((1024, 512), DATA / "cache1024" / "dentex_train"),
                 ((512, 256), DATA / "cache512" / "dentex_train")]:
    out.mkdir(parents=True, exist_ok=True)
    done = 0
    for n in names:
        dst = out / (Path(n).stem + ".jpg")
        if dst.exists():
            done += 1
            continue
        Image.open(src / n).convert("RGB").resize(size, Image.LANCZOS).save(dst, quality=90)
        done += 1
    print(f"dentex_train {size}: {done} cached")

# DENTEX val (50 PNGs)
cache_dir(DATA / "validation_data" / "quadrant_enumeration_disease" / "xrays",
          "val_*.png", DATA / "cache1024" / "dentex", (1024, 512))
cache_dir(DATA / "validation_data" / "quadrant_enumeration_disease" / "xrays",
          "val_*.png", DATA / "cache512" / "dentex", (512, 256))
# Tufts (1,000 JPGs)
cache_dir(DATA / "Radiographs", "*.JPG", DATA / "cache1024" / "tufts", (1024, 512))
cache_dir(DATA / "Radiographs", "*.JPG", DATA / "cache512" / "tufts", (512, 256))
print("caches done.")


In [ ]:
# 7b — Build the caches (~5 min).
!python colab/build_caches.py


## 7c — Rebuild the DENTEX validation triple

The DENTEX validation ground truth has no downloadable triple in its
Zenodo record, and the old source URL for `validation_triple2.json`
returns 404. So this cell rebuilds `data/validation_triple2.json` from
the project's own committed harmonised outputs
(`results/dentex_val_labels.csv` / `results/dentex_val_boxes.csv`,
already in the repo) — the same ground truth run 1 used — and verifies
it reproduces exactly 50 images / 182 boxes. `src/harmonise.py`
(cell 8) needs this file, so this cell MUST run before it. If the
verification fails, stop: do not train on a wrong validation set.

In [ ]:
# 7c — Rebuild data/validation_triple2.json from the committed harmonised CSVs.
import csv, json
from pathlib import Path

DIAG = ["Impacted", "Caries", "Periapical Lesion", "Deep Caries"]
labels = list(csv.DictReader(open("results/dentex_val_labels.csv")))
boxes = list(csv.DictReader(open("results/dentex_val_boxes.csv")))
images, anns = [], []
for i, r in enumerate(labels, start=1):
    images.append({"height": int(r["img_h"]), "width": int(r["img_w"]),
                   "id": i, "file_name": r["image"]})
img_id = {r["image"]: i + 1 for i, r in enumerate(labels)}
for j, b in enumerate(boxes, start=1):
    anns.append({"iscrowd": 0, "image_id": img_id[b["image"]],
                 "bbox": [float(b["x_min"]), float(b["y_min"]),
                          float(b["x_max"]) - float(b["x_min"]),
                          float(b["y_max"]) - float(b["y_min"])],
                 "segmentation": [], "id": j,
                 "area": (float(b["x_max"]) - float(b["x_min"])) * (float(b["y_max"]) - float(b["y_min"])),
                 "category_id_1": int(b["quadrant"]) - 1,
                 "category_id_2": int(b["tooth_position"]) - 1,
                 "category_id_3": DIAG.index(b["diagnosis"])})
cats1 = [{"id": i, "name": str(i + 1), "supercategory": str(i + 1)} for i in range(4)]
cats2 = [{"id": i, "name": str(i + 1), "supercategory": str(i + 1)} for i in range(8)]
cats3 = [{"id": i, "name": d, "supercategory": d} for i, d in enumerate(DIAG)]
Path("data").mkdir(exist_ok=True)
json.dump({"images": images, "annotations": anns, "categories_1": cats1,
           "categories_2": cats2, "categories_3": cats3},
          open("data/validation_triple2.json", "w"), indent=2)
assert len(images) == 50 and len(anns) == 182, (len(images), len(anns))
print(f"rebuilt data/validation_triple2.json: {len(images)} images, {len(anns)} boxes — verified")

In [ ]:
# 8 — Label tables (same as the local run): Tufts EDA + harmonisation.
# Needs data/validation_triple2.json — rebuilt in cell 7c above.
# Regenerates results/tufts_labels.csv, dentex_val_labels.csv and the box CSVs.
!python src/eda_tufts.py && python src/harmonise.py
!ls -la results/*.csv | head


## 9 — Train the detector, run A (GPU): + Tufts negatives, 10 epochs

ResNet50-FPN unfrozen at 1024×512, 10 epochs, mixed precision, with 200
Tufts train-split normal images added as box-free negatives (total
1,458 steps/epoch: 705 DENTEX + 553 rare-class duplicates + 200
negatives). **Expected time:** T4 ~45–60 min, L4 ~25–35 min,
A100 ~15 min. A checkpoint is saved after every epoch
(`results/fasterrcnn_v2_ep{N}.pt`).
If the runtime disconnects, resume with (adjust the epoch and lr to the
last saved checkpoint — lr is 0.005 for epochs 0–1, 0.0005 for 2–3,
0.00005 from epoch 4):
`!python colab/train_detector_v2.py --resume results/fasterrcnn_v2_ep3.pt --start-epoch 4 --lr 0.00005 --epochs 10 --tufts-negatives 200`
and re-run the Drive-copy cell at the end so far.

In [ ]:
# 9 — Train the GPU detector, run A (long cell — let it run).
!python colab/train_detector_v2.py --epochs 10 --tufts-negatives 200

In [ ]:
# 9b — Detector run A results: compare with run 1 (mAP 0.355 @0.5, 0.377 @0.3).
import json
for f in ("results/detector_metrics_v2.json", "results/detector_metrics_v2_thr03.json"):
    print("==", f)
    print(json.dumps(json.load(open(f)), indent=2))

## 9c — Train the detector, run B (GPU): run A + focal loss

Same as run A, plus focal loss on the ROI classification head
(gamma 2.0, alpha 0.25) to target the periapical regression
(0.29 → 0.11 in run 1). Run this only if time permits in this session —
same time as run A (~45–60 min on a T4). NOTE: focal losses are on a
smaller scale than cross-entropy, so run-B training losses are NOT
comparable to run-A losses — compare the metrics, not the losses.
Checkpoints: `results/fasterrcnn_v2focal_ep{N}.pt`.

In [ ]:
# 9c — Train the GPU detector, run B: focal loss (long cell — let it run).
!python colab/train_detector_v2.py --epochs 10 --tufts-negatives 200 --focal --tag v2focal

In [ ]:
# 9d — Detector run B results: compare with run A and run 1.
import json
for f in ("results/detector_metrics_v2focal.json", "results/detector_metrics_v2focal_thr03.json"):
    print("==", f)
    print(json.dumps(json.load(open(f)), indent=2))

## 10 — Train the classifier on GPU: 3-seed replication

Same mixed-site ResNet18 recipe as run 1 (Tufts + 102 effective
DENTEX-train images, temperature scaling, tuned threshold), run with
seeds 42, 0 and 1. The seed controls the Tufts split, the DENTEX-train
sample, the init and the data order, so this replicates the whole
recipe. ~5–10 min per seed on a T4. The `--aggregate` step then writes
`results/classifier_metrics_v2_gpu.json` (per-seed entries + mean/std)
and `results/roc_v2_gpu.png`.

In [ ]:
# 10 — Train the GPU classifier, seeds 42, 0, 1, then aggregate.
!python colab/train_classifier_v2.py --seed 42
!python colab/train_classifier_v2.py --seed 0
!python colab/train_classifier_v2.py --seed 1
!python colab/train_classifier_v2.py --aggregate

In [ ]:
# 10b — Classifier replication results: compare with run 1 (DENTEX val AUC 0.554, Tufts test 0.587)
# and with the CPU v2 run (DENTEX val AUC 0.84 — the number this replication tests).
import json
d = json.load(open("results/classifier_metrics_v2_gpu.json"))
print(json.dumps(d["summary"], indent=2))
for s, per in d["per_seed"].items():
    print(f"seed {s}: DENTEX val AUC {per['dentex_val']['at_0.5_raw']['auc']:.3f}, "
          f"Tufts test AUC {per['tufts_test']['at_0.5_raw']['auc']:.3f}, "
          f"Tufts oracle thr {per['tufts_test']['oracle_threshold']['threshold']:.2f}, "
          f"DENTEX oracle thr {per['dentex_val']['oracle_threshold']['threshold']:.2f}")
print(json.dumps(d["per_seed"]["42"]["cross_site_thresholds"], indent=2))

## 11 — Save everything to Google Drive

Copies the new v2 metrics, predictions, figures, checkpoints and logs to
`MyDrive/dental-panoramic-ai/colab_run_<date>_v2/`. Run this even if a
training cell failed part-way — the per-epoch checkpoints are what
matter. Run-1 files (`colab_run_2026-11-07/`) are NOT touched. Then copy
the new numbers into `manuscript/draft.md` by hand (do not paste
unreviewed numbers into the manuscript from here) — and report all
three runs honestly, including any non-reproduction.

In [ ]:
# 11 — Copy all v2 outputs to Drive.
import datetime, os, shutil
today = datetime.date.today().isoformat()
dest = f"/content/drive/MyDrive/dental-panoramic-ai/colab_run_{today}_v2"
os.makedirs(dest, exist_ok=True)
patterns = ("detector_metrics_v2*.json", "classifier_metrics_v2*.json",
            "detector_predictions_v2*.csv", "classifier_predictions_v2_gpu*.csv",
            "detector_val_boxes_v2*.csv", "detector_examples_v2*.png",
            "roc_v2*.png", "tufts_detector_negatives_v2.csv",
            "fasterrcnn_v2_ep*.pt", "fasterrcnn_v2focal_ep*.pt",
            "resnet18_mixed_v2_gpu*.pt",
            "detector_v2_train.log", "detector_v2focal_train.log")
for pattern in patterns:
    for f in sorted(__import__("glob").glob(f"results/{pattern}")):
        shutil.copy(f, dest)
        print("saved:", os.path.join(dest, os.path.basename(f)))
print("\nAll v2 outputs are in:", dest)